# Assignment 4 - Lab 4

## Vision Transformer Feature Preparation for Image Captioning

This notebook completes **Task 1** and **Task 2** from the assignment PDF.

- **Task 1:** Load Flickr8k, clean captions, tokenize captions, build a vocabulary, and convert captions to token IDs.
- **Task 2:** Load a frozen pre-trained ViT encoder, preprocess images to `224 x 224`, and extract image memory using both required strategies: patch tokens and CLS token.

Every code cell includes line-by-line comments to explain what each line is doing.


## Environment Setup

Run the installation cell only if the required packages are missing. The rest of the notebook assumes `torch`, `datasets`, `transformers`, `Pillow`, `matplotlib`, and `tqdm` are available.


In [ ]:
# This line is commented out so packages are not reinstalled every time the notebook runs.
# %pip install -q torch torchvision transformers datasets pillow matplotlib tqdm


In [ ]:
# Import math so we can compute the ViT patch attention grid size later.
import math
# Import random so we can set a fixed random seed for reproducible sampling.
import random
# Import re so we can remove repeated whitespace while cleaning captions.
import re
# Import string so we can access the standard punctuation characters.
import string
# Import Counter so we can count word frequencies for the vocabulary.
from collections import Counter
# Import Dict so function signatures can clearly describe dictionary inputs.
from typing import Dict
# Import List so function signatures can clearly describe list outputs.
from typing import List
# Import Sequence so image preprocessing can accept any ordered group of images.
from typing import Sequence

# Import matplotlib so we can display images and ViT attention heatmaps.
import matplotlib.pyplot as plt
# Import torch as the main deep-learning library.
import torch
# Import torch.nn so we can define projection layers inside an nn.Module.
import torch.nn as nn
# Import load_dataset so we can download/load Flickr8k from Hugging Face.
from datasets import load_dataset
# Import PIL Image so we can check and convert image objects to RGB.
from PIL import Image
# Import Dataset so we can create a custom PyTorch dataset for image-caption pairs.
from torch.utils.data import Dataset
# Import DataLoader so we can batch image-caption examples.
from torch.utils.data import DataLoader
# Import tqdm so long loops show progress bars in the notebook.
from tqdm.auto import tqdm
# Import AutoImageProcessor so images are resized and normalized exactly as ViT expects.
from transformers import AutoImageProcessor
# Import AutoModel so we can load the pre-trained ViT encoder.
from transformers import AutoModel

# Store a fixed seed value so random behavior is repeatable.
SEED = 42
# Seed Python's random module for reproducible Python-level randomness.
random.seed(SEED)
# Seed PyTorch for reproducible tensor operations where possible.
torch.manual_seed(SEED)

# Select the GPU when CUDA is available; otherwise use the CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Display the selected device so we know where tensors and models will run.
DEVICE


# Task 1 - Data Preparation

The assignment asks us to load Flickr8k images and captions, normalize captions, tokenize them, build a limited-size vocabulary, and add special tokens: `<SOS>`, `<EOS>`, `<PAD>`, and `<UNK>`.


In [ ]:
# Store the Hugging Face dataset name for Flickr8k.
DATASET_NAME = "jxie/flickr8k"
# Limit the vocabulary to the assignment-suggested maximum size.
MAX_VOCAB_SIZE = 10_000
# Keep words that appear at least this many times in the training captions.
MIN_WORD_FREQ = 1
# Set the fixed caption length used for padding and truncation.
MAX_CAPTION_LEN = 40
# Set the number of image-caption pairs processed in each batch.
BATCH_SIZE = 16

# Store the dataset column name that contains the image.
IMAGE_COLUMN = "image"
# Store the padding token used to fill unused caption positions.
PAD_TOKEN = "<PAD>"
# Store the start-of-sentence token placed before every caption.
SOS_TOKEN = "<SOS>"
# Store the end-of-sentence token placed after every caption.
EOS_TOKEN = "<EOS>"
# Store the unknown-word token used for words outside the vocabulary.
UNK_TOKEN = "<UNK>"
# Store the special tokens in a fixed order so their indices stay stable.
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]


In [ ]:
# Define a helper that lowercases captions, removes punctuation, and normalizes spaces.
def normalize_caption(caption: str) -> str:
    # Convert the input to a string in case the dataset returns a non-string value.
    caption = str(caption)
    # Lowercase the caption and remove spaces at the beginning and end.
    caption = caption.lower().strip()
    # Remove punctuation marks such as commas, periods, and quotation marks.
    caption = caption.translate(str.maketrans("", "", string.punctuation))
    # Replace one or more whitespace characters with a single normal space.
    caption = re.sub(r"\s+", " ", caption)
    # Return the cleaned caption text.
    return caption

# Define a helper that converts one caption string into word tokens.
def tokenize_caption(caption: str) -> List[str]:
    # Clean the caption before tokenizing so vocabulary words are consistent.
    normalized = normalize_caption(caption)
    # Split the cleaned caption on spaces, or return an empty list for an empty caption.
    return normalized.split() if normalized else []

# Print a confirmation message after the helper functions are defined.
print("Caption cleaning and tokenization helpers are ready.")


In [ ]:
# Load the Flickr8k dataset from Hugging Face.
dataset = load_dataset(DATASET_NAME)
# Print the dataset object so we can verify the available splits and row counts.
print(dataset)

# Read the column names from the training split because all splits should share the same schema.
train_columns = dataset["train"].column_names
# Ensure the image column exists before building datasets that depend on it.
assert IMAGE_COLUMN in train_columns, f"Missing required image column: {IMAGE_COLUMN}"

# Detect caption columns for datasets that store captions as caption_0, caption_1, ..., caption_4.
numbered_caption_columns = [column for column in train_columns if column.startswith("caption_")]
# Detect the common alternative schema where all captions are stored in one column named caption.
single_caption_column = "caption" if "caption" in train_columns else None
# Use numbered caption columns when available; otherwise use the single caption column.
CAPTION_COLUMNS = numbered_caption_columns if numbered_caption_columns else ([single_caption_column] if single_caption_column else [])
# Ensure at least one caption column was found before continuing.
assert CAPTION_COLUMNS, "No caption columns were found in the dataset."

# Check every split to make sure the required image and caption columns are present.
for split_name in ["train", "validation", "test"]:
    # Loop through the image column plus all detected caption columns.
    for column in [IMAGE_COLUMN] + CAPTION_COLUMNS:
        # Raise a clear error if the expected column is missing from the split.
        assert column in dataset[split_name].column_names, f"Missing {column} in {split_name} split"

# Print the detected caption columns for transparency.
print("Detected caption columns:", CAPTION_COLUMNS)


In [ ]:
# Define a helper that returns all captions from one dataset row.
def get_row_captions(row) -> List[str]:
    # Create an empty list that will hold the captions for this image.
    captions = []
    # Loop through every detected caption column.
    for caption_column in CAPTION_COLUMNS:
        # Read the value stored in the current caption column.
        value = row[caption_column]
        # If the value is a list, extend the captions list with all captions in that list.
        if isinstance(value, list):
            # Convert each list element to a string and add it to the caption list.
            captions.extend(str(caption) for caption in value)
        # If the value is not a list, treat it as one caption string.
        else:
            # Convert the value to a string and add it to the caption list.
            captions.append(str(value))
    # Return all captions associated with this image row.
    return captions

# Create a Counter object that will store word frequencies from training captions only.
word_counts = Counter()
# Loop over every image row in the training split with a progress bar.
for row in tqdm(dataset["train"], desc="Counting words in training captions"):
    # Loop over the five human captions associated with this image.
    for caption in get_row_captions(row):
        # Convert the caption into cleaned word tokens.
        tokens = tokenize_caption(caption)
        # Add the tokens from this caption to the global word-frequency counter.
        word_counts.update(tokens)

# Create the word-to-index dictionary with the required special tokens first.
word_to_idx = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}
# Loop over words from most frequent to least frequent.
for word, freq in word_counts.most_common():
    # Skip words that do not satisfy the minimum frequency threshold.
    if freq < MIN_WORD_FREQ:
        # Continue to the next word without adding this rare word.
        continue
    # Stop adding words when the vocabulary reaches the maximum allowed size.
    if len(word_to_idx) >= MAX_VOCAB_SIZE:
        # Exit the loop because the vocabulary is full.
        break
    # Assign the current word the next available integer index.
    word_to_idx[word] = len(word_to_idx)

# Create the reverse dictionary so token IDs can be converted back to words later.
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
# Store the integer index used for padding tokens.
PAD_IDX = word_to_idx[PAD_TOKEN]
# Store the integer index used for start-of-sentence tokens.
SOS_IDX = word_to_idx[SOS_TOKEN]
# Store the integer index used for end-of-sentence tokens.
EOS_IDX = word_to_idx[EOS_TOKEN]
# Store the integer index used for unknown-word tokens.
UNK_IDX = word_to_idx[UNK_TOKEN]

# Print the final vocabulary size.
print(f"Vocabulary size: {len(word_to_idx)}")
# Display the first twenty vocabulary entries as a quick sanity check.
list(word_to_idx.items())[:20]


In [ ]:
# Define a helper that converts a caption string into a fixed-length list of token IDs.
def numericalize_caption(caption: str, word_to_idx: Dict[str, int], max_len: int) -> List[int]:
    # Tokenize the caption using the cleaning rules from Task 1.
    tokens = tokenize_caption(caption)
    # Truncate the caption so there is still room for <SOS> and <EOS>.
    tokens = tokens[: max_len - 2]
    # Start the numerical caption with the <SOS> token index.
    token_ids = [SOS_IDX]
    # Convert each word token to its vocabulary index, using <UNK> for unseen words.
    token_ids.extend(word_to_idx.get(token, UNK_IDX) for token in tokens)
    # Append the <EOS> token index to mark the end of the caption.
    token_ids.append(EOS_IDX)
    # Add <PAD> indices until the caption reaches the fixed maximum length.
    token_ids.extend([PAD_IDX] * (max_len - len(token_ids)))
    # Return the padded list of caption token IDs.
    return token_ids

# Define a PyTorch dataset that expands each Flickr8k image row into image-caption examples.
class Flickr8kCaptionDataset(Dataset):
    # Initialize the dataset with a Hugging Face split and vocabulary information.
    def __init__(self, hf_split, word_to_idx: Dict[str, int], max_caption_len: int):
        # Store the Hugging Face split so rows can be read later.
        self.hf_split = hf_split
        # Store the vocabulary dictionary for converting captions to IDs.
        self.word_to_idx = word_to_idx
        # Store the maximum caption length for padding and truncation.
        self.max_caption_len = max_caption_len
        # Create an empty index that maps each dataset item to one image row and one caption.
        self.index = []
        # Loop through every row in the Hugging Face split.
        for row_idx in range(len(hf_split)):
            # Read every caption associated with this image row.
            captions = get_row_captions(hf_split[row_idx])
            # Loop through the caption positions for this image row.
            for caption_idx in range(len(captions)):
                # Store the row index and caption index as one training example.
                self.index.append((row_idx, caption_idx))

    # Return the number of image-caption examples in this expanded dataset.
    def __len__(self) -> int:
        # The index contains one entry for every image-caption pair.
        return len(self.index)

    # Return one image-caption example by integer index.
    def __getitem__(self, idx: int):
        # Look up which image row and caption position correspond to this item.
        row_idx, caption_idx = self.index[idx]
        # Load the full row from the Hugging Face split.
        row = self.hf_split[row_idx]
        # Read the image object from the row.
        image = row[IMAGE_COLUMN]
        # Read all captions for this row.
        captions = get_row_captions(row)
        # Select the caption assigned to this dataset item.
        caption = captions[caption_idx]
        # Convert PIL images to RGB because ViT expects three-channel RGB inputs.
        if isinstance(image, Image.Image):
            # Convert grayscale or other image modes to RGB.
            image = image.convert("RGB")
        # Convert the caption text to a padded tensor of token IDs.
        token_ids = torch.tensor(numericalize_caption(caption, self.word_to_idx, self.max_caption_len), dtype=torch.long)
        # Remove the last token to create decoder inputs for teacher forcing.
        input_ids = token_ids[:-1]
        # Remove the first token to create decoder targets for teacher forcing.
        target_ids = token_ids[1:]
        # Return the image, raw caption, decoder input IDs, and decoder target IDs.
        return {"image": image, "caption": caption, "input_ids": input_ids, "target_ids": target_ids}

# Define a collate function because PIL images cannot be stacked like tensors before preprocessing.
def caption_collate_fn(batch):
    # Return a dictionary containing batched images, captions, input IDs, and target IDs.
    return {
        # Keep images as a Python list so AutoImageProcessor can preprocess them together.
        "images": [item["image"] for item in batch],
        # Keep raw captions as a Python list for debugging and later qualitative examples.
        "captions": [item["caption"] for item in batch],
        # Stack all decoder input tensors into one batch tensor.
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        # Stack all decoder target tensors into one batch tensor.
        "target_ids": torch.stack([item["target_ids"] for item in batch]),
    }


In [ ]:
# Build the expanded training dataset with one item per image-caption pair.
train_caption_dataset = Flickr8kCaptionDataset(dataset["train"], word_to_idx, MAX_CAPTION_LEN)
# Build the expanded validation dataset with one item per image-caption pair.
val_caption_dataset = Flickr8kCaptionDataset(dataset["validation"], word_to_idx, MAX_CAPTION_LEN)
# Build the expanded test dataset with one item per image-caption pair.
test_caption_dataset = Flickr8kCaptionDataset(dataset["test"], word_to_idx, MAX_CAPTION_LEN)

# Create a shuffled DataLoader for training examples.
train_loader = DataLoader(train_caption_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=caption_collate_fn)
# Create a non-shuffled DataLoader for validation examples.
val_loader = DataLoader(val_caption_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=caption_collate_fn)
# Create a non-shuffled DataLoader for test examples.
test_loader = DataLoader(test_caption_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=caption_collate_fn)

# Read one sample batch to verify that the data pipeline works.
sample_batch = next(iter(train_loader))
# Print the number of expanded training image-caption pairs.
print(f"Number of train image-caption pairs: {len(train_caption_dataset)}")
# Print the shape of decoder input IDs to confirm batching and padding.
print(f"Caption decoder input shape: {sample_batch['input_ids'].shape}")
# Print the shape of decoder target IDs to confirm the shifted target sequence.
print(f"Caption decoder target shape: {sample_batch['target_ids'].shape}")
# Print one raw caption so we can visually confirm the dataset was loaded correctly.
print(f"Example raw caption: {sample_batch['captions'][0]}")
# Print the token IDs for the same example caption.
print(f"Example token IDs: {sample_batch['input_ids'][0].tolist()}")


# Task 2 - ViT Encoder Feature Extraction

The assignment asks us to load a pre-trained ViT, resize/preprocess images to `224 x 224`, keep ViT frozen initially, and implement both encoder-memory strategies:

- **Strategy A - Patch tokens:** remove the CLS token and use all 196 patch tokens, giving `[batch, 196, 768]` before projection.
- **Strategy B - CLS token:** use only the CLS token, giving `[batch, 1, 768]` before projection.

Both strategies are projected from ViT hidden size `768` to the decoder dimension `d_model`.


In [ ]:
# Store the exact pre-trained ViT checkpoint required for feature extraction.
VIT_MODEL_NAME = "google/vit-base-patch16-224"
# Store the Transformer decoder dimension that ViT features will be projected into.
D_MODEL = 512

# Load the image processor that performs ViT resizing, normalization, and tensor conversion.
image_processor = AutoImageProcessor.from_pretrained(VIT_MODEL_NAME)
# Load the pre-trained ViT encoder and ask it to return attention maps for visualization.
vit_encoder = AutoModel.from_pretrained(VIT_MODEL_NAME, output_attentions=True).to(DEVICE)

# Loop through every parameter tensor in the ViT encoder.
for parameter in vit_encoder.parameters():
    # Disable gradient updates so the ViT stays frozen during initial training.
    parameter.requires_grad = False
# Put the ViT in evaluation mode so dropout and similar layers behave deterministically.
vit_encoder.eval()

# Print the checkpoint name that was loaded.
print(f"Loaded {VIT_MODEL_NAME}")
# Print the ViT hidden size, which should be 768 for ViT-base.
print(f"ViT hidden size: {vit_encoder.config.hidden_size}")
# Print the patch size, which should be 16 for patch16 models.
print(f"Patch size: {vit_encoder.config.patch_size}")
# Print the expected image size, which should be 224.
print(f"Image size: {vit_encoder.config.image_size}")


In [ ]:
# Define a module that wraps frozen ViT feature extraction plus trainable projection layers.
class ViTFeatureExtractor(nn.Module):
    # Initialize the feature extractor with a ViT model and decoder dimension.
    def __init__(self, vit_model: nn.Module, d_model: int):
        # Initialize the parent nn.Module class.
        super().__init__()
        # Store the frozen ViT encoder.
        self.vit = vit_model
        # Store the ViT hidden size, which is 768 for google/vit-base-patch16-224.
        self.hidden_size = vit_model.config.hidden_size
        # Create a trainable projection for patch-token memory: 768 -> d_model.
        self.patch_projection = nn.Linear(self.hidden_size, d_model)
        # Create a trainable projection for CLS-token memory: 768 -> d_model.
        self.cls_projection = nn.Linear(self.hidden_size, d_model)

    # Define the forward pass for extracting either patch-token or CLS-token memory.
    def forward(self, pixel_values: torch.Tensor, strategy: str = "patch"):
        # Stop gradient tracking while running the frozen ViT encoder.
        with torch.no_grad():
            # Run the ViT on preprocessed image tensors and keep attention outputs.
            outputs = self.vit(pixel_values=pixel_values, output_attentions=True)
            # Read the final hidden states with shape [batch, 197, 768].
            last_hidden_state = outputs.last_hidden_state

        # Check whether the requested strategy is the patch-token strategy.
        if strategy == "patch":
            # Drop token 0 because it is CLS, leaving the 196 spatial patch tokens.
            patch_tokens = last_hidden_state[:, 1:, :]
            # Project patch tokens from 768 dimensions to the decoder dimension.
            memory = self.patch_projection(patch_tokens)
        # Check whether the requested strategy is the CLS-only baseline.
        elif strategy == "cls":
            # Keep only token 0, which is the global CLS image representation.
            cls_token = last_hidden_state[:, :1, :]
            # Project the CLS token from 768 dimensions to the decoder dimension.
            memory = self.cls_projection(cls_token)
        # Reject invalid strategy names with a clear error message.
        else:
            # Raise an exception so the caller knows the strategy argument is invalid.
            raise ValueError("strategy must be either 'patch' or 'cls'")

        # Return projected memory and raw ViT outputs for later attention visualization.
        return memory, outputs

# Create the ViT feature extractor and move its projection layers to the selected device.
feature_extractor = ViTFeatureExtractor(vit_encoder, D_MODEL).to(DEVICE)
# Print the module so the two projection layers are visible in the notebook.
print(feature_extractor)


In [ ]:
# Define a helper that converts PIL images into ViT-ready pixel tensors.
def preprocess_images_for_vit(images: Sequence[Image.Image]) -> torch.Tensor:
    # Use the Hugging Face image processor to resize images to 224 x 224 and normalize them.
    processed = image_processor(images=list(images), return_tensors="pt")
    # Move the resulting [batch, 3, 224, 224] tensor to the selected device.
    return processed["pixel_values"].to(DEVICE)

# Preprocess the images from the sample batch for the ViT encoder.
pixel_values = preprocess_images_for_vit(sample_batch["images"])
# Extract Strategy A memory using all patch tokens except CLS.
patch_memory, patch_outputs = feature_extractor(pixel_values, strategy="patch")
# Extract Strategy B memory using only the CLS token.
cls_memory, cls_outputs = feature_extractor(pixel_values, strategy="cls")

# Print the preprocessed image tensor shape.
print(f"Preprocessed images: {pixel_values.shape}")
# Print the raw ViT token output shape before selecting patch or CLS memory.
print(f"Raw ViT output tokens: {patch_outputs.last_hidden_state.shape}")
# Print the projected patch-token memory shape.
print(f"Strategy A patch memory: {patch_memory.shape}  # expected [B, 196, {D_MODEL}]")
# Print the projected CLS-token memory shape.
print(f"Strategy B CLS memory:   {cls_memory.shape}  # expected [B, 1, {D_MODEL}]")

# Verify that images were converted to [channels, height, width] = [3, 224, 224].
assert pixel_values.shape[1:] == (3, 224, 224)
# Verify that ViT-base returned 197 tokens: 1 CLS token plus 196 patch tokens.
assert patch_outputs.last_hidden_state.shape[1:] == (197, 768)
# Verify that Strategy A returned 196 projected patch tokens per image.
assert patch_memory.shape[1:] == (196, D_MODEL)
# Verify that Strategy B returned one projected CLS token per image.
assert cls_memory.shape[1:] == (1, D_MODEL)


In [ ]:
# Define a helper that extracts ViT features for every batch in a DataLoader.
def extract_vit_features_for_loader(loader: DataLoader, strategy: str, max_batches: int | None = None):
    # Put the feature extractor in evaluation mode during feature extraction.
    feature_extractor.eval()
    # Create an empty list to store extracted feature batches.
    collected = []
    # Loop through batches from the loader with a progress bar.
    for batch_idx, batch in enumerate(tqdm(loader, desc=f"Extracting {strategy} ViT features")):
        # Stop early when max_batches is provided and the limit has been reached.
        if max_batches is not None and batch_idx >= max_batches:
            # Exit the loop to avoid extracting more batches.
            break
        # Preprocess the batch images into ViT pixel tensors.
        pixels = preprocess_images_for_vit(batch["images"])
        # Extract projected ViT memory using the requested strategy.
        memory, _ = feature_extractor(pixels, strategy=strategy)
        # Store CPU copies of memory and caption tensors so they can be reused later.
        collected.append({
            # Store projected visual memory for the decoder.
            "memory": memory.detach().cpu(),
            # Store teacher-forcing decoder inputs.
            "input_ids": batch["input_ids"].cpu(),
            # Store teacher-forcing decoder targets.
            "target_ids": batch["target_ids"].cpu(),
            # Store raw captions for qualitative inspection.
            "captions": batch["captions"],
        })
    # Return the list of extracted feature batches.
    return collected

# Extract one batch with Strategy A as a quick demonstration.
one_patch_batch = extract_vit_features_for_loader(train_loader, strategy="patch", max_batches=1)
# Extract one batch with Strategy B as a quick demonstration.
one_cls_batch = extract_vit_features_for_loader(train_loader, strategy="cls", max_batches=1)

# Print the Strategy A feature tensor shape for the demonstration batch.
print(one_patch_batch[0]["memory"].shape)
# Print the Strategy B feature tensor shape for the demonstration batch.
print(one_cls_batch[0]["memory"].shape)


In [ ]:
# Define a helper that converts CLS-to-patch attention into a square grid.
def cls_to_patch_attention_grid(vit_outputs, image_index: int = 0) -> torch.Tensor:
    # Read the last ViT layer attention for one image with shape [heads, 197, 197].
    last_layer_attention = vit_outputs.attentions[-1][image_index]
    # Select attention from the CLS query token to all 196 patch key tokens.
    cls_attention = last_layer_attention[:, 0, 1:]
    # Average attention scores across all attention heads.
    mean_attention = cls_attention.mean(dim=0)
    # Compute the side length of the square patch grid, which is 14 for 196 patches.
    grid_size = int(math.sqrt(mean_attention.numel()))
    # Reshape the 196 attention values into a [14, 14] grid and move it to CPU.
    return mean_attention.reshape(grid_size, grid_size).detach().cpu()

# Define a helper that overlays a ViT attention grid on top of an image.
def show_vit_attention_overlay(image: Image.Image, attention_grid: torch.Tensor, title: str = "ViT CLS-to-patch attention"):
    # Convert the image to RGB so matplotlib displays it consistently.
    image = image.convert("RGB")
    # Convert the attention tensor to a NumPy array for matplotlib.
    attention = attention_grid.numpy()
    # Normalize attention values to the range [0, 1] for stable visualization.
    attention = (attention - attention.min()) / (attention.max() - attention.min() + 1e-8)
    # Create a figure with one panel for the original image and one panel for the overlay.
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    # Show the original image in the first panel.
    axes[0].imshow(image)
    # Title the first panel.
    axes[0].set_title("Original image")
    # Hide the first panel axes for a cleaner visualization.
    axes[0].axis("off")
    # Show the original image again in the second panel.
    axes[1].imshow(image)
    # Overlay the normalized attention heatmap across the full image extent.
    axes[1].imshow(attention, cmap="jet", alpha=0.45, extent=(0, image.width, image.height, 0))
    # Title the second panel.
    axes[1].set_title(title)
    # Hide the second panel axes for a cleaner visualization.
    axes[1].axis("off")
    # Adjust spacing so titles and images do not overlap.
    plt.tight_layout()
    # Display the completed visualization.
    plt.show()

# Convert ViT attention from the first sample image into a 14 x 14 grid.
attention_grid = cls_to_patch_attention_grid(patch_outputs, image_index=0)
# Show the original image beside its ViT attention heatmap overlay.
show_vit_attention_overlay(sample_batch["images"][0], attention_grid)


## Task 1 and Task 2 Checklist

Task 1 completed:

- Loaded Flickr8k with train/validation/test handling.
- Lowercased captions and removed punctuation.
- Tokenized captions.
- Built a vocabulary with max size `10,000`.
- Added `<SOS>`, `<EOS>`, `<PAD>`, and `<UNK>`.
- Converted captions to padded token IDs and prepared teacher-forcing inputs/targets.

Task 2 completed:

- Loaded frozen `google/vit-base-patch16-224`.
- Preprocessed images to ViT-compatible `224 x 224` tensors.
- Implemented Strategy A using patch tokens with output `[B, 196, 768]` before projection and `[B, 196, D_MODEL]` after projection.
- Implemented Strategy B using CLS token with output `[B, 1, 768]` before projection and `[B, 1, D_MODEL]` after projection.
- Added linear projections from `768 -> D_MODEL`.
- Added a batch feature-extraction utility for later decoder training.
- Added a patch-token attention heatmap helper to support visual inspection.
